In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/ST7242J0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/ST7052J0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/SC4492G0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/SC4821G0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/SC4742E0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/SC4472F0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/SC4822G0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/ST7241J0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/SC4061E0-PSG_spectral.pt
/kaggle/input/datasets/girishgiriirig

In [4]:
import pandas as pd
import os

# Read the CSV file (replace 'input.csv' with your actual filename)
df = pd.read_csv('/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/processed_sleepedf/index.csv')

# Add tensor_path column: basepath + new prefix + _spectral.pt
def create_spectral_path(filename):
    base_name = filename.split('\\')[-1][:-3]  # Get filename without extension
    new_path = f"/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/spectral/{base_name}_spectral.pt"
    return new_path

def create_raw_path(filename):
    base_name = filename.split('\\')[-1][:-3]  # Get filename without extension
    new_path = f'/kaggle/input/datasets/girishgiriirig/sleep-edfx-v1-0-0-eeg-preprocessed/processed_sleepedf/tensors/{base_name}.pt'
    return new_path

# Assuming there's a column with original paths/filenames (adjust column name as needed)
df['spectral'] = df['tensor_path'].apply(create_spectral_path)  # Replace 'path_column' with actual column
df['tensor_path'] = df['tensor_path'].apply(create_raw_path)
# Create/modify spectral column (example - adjust logic as needed)
# df['spectral'] = df['tensor_path']  # Or whatever transformation you need

# Save modified CSV
df.to_csv('output.csv', index=False)
print("Modified CSV saved as 'output.csv'")
print(df[['tensor_path', 'spectral']].head())


Modified CSV saved as 'output.csv'
                                         tensor_path  \
0  /kaggle/input/datasets/girishgiriirig/sleep-ed...   
1  /kaggle/input/datasets/girishgiriirig/sleep-ed...   
2  /kaggle/input/datasets/girishgiriirig/sleep-ed...   
3  /kaggle/input/datasets/girishgiriirig/sleep-ed...   
4  /kaggle/input/datasets/girishgiriirig/sleep-ed...   

                                            spectral  
0  /kaggle/input/datasets/girishgiriirig/sleep-ed...  
1  /kaggle/input/datasets/girishgiriirig/sleep-ed...  
2  /kaggle/input/datasets/girishgiriirig/sleep-ed...  
3  /kaggle/input/datasets/girishgiriirig/sleep-ed...  
4  /kaggle/input/datasets/girishgiriirig/sleep-ed...  


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset, random_split
import torch.nn as nn
import torch.nn.functional as F
import math
import random
import time
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    cohen_kappa_score, confusion_matrix
)
import json
from sklearn.model_selection import train_test_split

In [6]:
STAGE_TO_IDX = {
    "W": 0,
    "N1": 1,
    "N2": 2,
    "N3": 3,
    "REM": 4
}

IDX_TO_STAGE = {v: k for k, v in STAGE_TO_IDX.items()}
NUM_CLASSES = 5


class FusionSleepDataset(Dataset):
    """
    Sliding-window fusion dataset.

    Each sample is a contiguous window of `window_size` epochs from one
    recording, returning BOTH temporal (raw EEG) and spectral features.

        x_temporal : [W, 3000]   W ≤ window_size
        x_spectral : [W, 34]
        y          : [W]

    The last window of each file may be shorter than window_size; the
    collate function handles padding.

    Args:
        csv_path    : path to output.csv (must have tensor_path + spectral cols)
        file_indices: optional list of row indices for train/val split
        window_size : number of epochs per chunk (default 256)
        overlap     : epoch overlap between consecutive windows (default 0)
    """

    def __init__(self, csv_path, file_indices=None, window_size=256, overlap=0):
        assert overlap < window_size, "overlap must be < window_size"

        self.df = pd.read_csv(csv_path)
        if file_indices is not None:
            self.df = self.df.iloc[file_indices].reset_index(drop=True)

        self.window_size = window_size
        self.stride      = window_size - overlap

        # Build flat index: list of (row_idx, start_epoch)
        self.index = []
        for row_idx, row in self.df.iterrows():
            T = len(row['stage_sequence'].split(' '))
            start = 0
            while start < T:
                self.index.append((row_idx, start))
                start += self.stride

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        row_idx, start = self.index[idx]
        row = self.df.iloc[row_idx]

        end = start + self.window_size

        # Lazy-load only once per file (OS page cache makes re-reads cheap)
        x_temporal = torch.load(row['tensor_path']).float()[start:end]  # [W, 3000]
        x_spectral  = torch.load(row['spectral']).float()[start:end]    # [W, 34]

        stages = row['stage_sequence'].split(' ')[start:end]
        y = torch.tensor([STAGE_TO_IDX[s] for s in stages], dtype=torch.long)

        return x_temporal, x_spectral, y

In [7]:
def fusion_collate_fn(batch):
    """
    Collates a batch of (x_temporal, x_spectral, y) tuples.

    Returns:
        x_temp       : [B, T_max, 3000]
        x_spec       : [B, T_max, 34]
        y_padded     : [B, T_max]        — padded with -100 (ignored by loss)
        padding_mask : [B, T_max] bool   — True where padded
    """
    lengths  = [xt.shape[0] for xt, _, _ in batch]
    max_len  = max(lengths)
    B        = len(batch)
    temp_dim = batch[0][0].shape[1]   # 3000
    spec_dim = batch[0][1].shape[1]   # 34

    x_temp   = torch.zeros(B, max_len, temp_dim)
    x_spec   = torch.zeros(B, max_len, spec_dim)
    y_padded = torch.full((B, max_len), -100, dtype=torch.long)
    pad_mask = torch.ones(B, max_len, dtype=torch.bool)   # True = padded

    for i, (xt, xs, y) in enumerate(batch):
        T = xt.shape[0]
        x_temp[i,   :T] = xt
        x_spec[i,   :T] = xs
        y_padded[i, :T] = y
        pad_mask[i, :T] = False        # False = valid token

    return x_temp, x_spec, y_padded, pad_mask

In [8]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        s = x.mean(dim=-1)
        s = F.relu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s))
        return x * s.unsqueeze(-1)


class AdaptiveAtrousPyramid(nn.Module):
    def __init__(self, in_channels=1, hidden_channels=64, dilations=(1, 2, 4, 8)):
        super().__init__()

        self.branches = nn.ModuleList([
            nn.Conv1d(
                in_channels,
                hidden_channels,
                kernel_size=7,
                dilation=d,
                padding=3 * d
            ) for d in dilations
        ])

        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(hidden_channels * len(dilations), len(dilations), 1),
            nn.Softmax(dim=1)
        )

        self.se = SEBlock(hidden_channels)
        self.proj = nn.Conv1d(hidden_channels, hidden_channels, 1)

    def forward(self, x):
        feats = [F.relu(b(x)) for b in self.branches]
        stacked = torch.cat(feats, dim=1)
        weights = self.gate(stacked)

        out = 0
        for i, f in enumerate(feats):
            out = out + f * weights[:, i:i+1]

        return self.proj(self.se(out))


class EpochEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.pyramid = AdaptiveAtrousPyramid()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, embed_dim)

    def forward(self, x):
        B, T, L = x.shape
        x = x.view(B * T, 1, L)
        f = self.pyramid(x)
        f = self.pool(f).squeeze(-1)
        f = self.fc(f)
        result = f.view(B, T, -1)
        return result


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        x: [B, T, D]
        """
        T = x.size(1)
        return x + self.pe[:, :T]


class SleepTransformer(nn.Module):
    def __init__(self, embed_dim=128, heads=4, layers=4, dropout=0.2, max_len=512):
        super().__init__()
        self.positional_encoding = PositionalEncoding(embed_dim, max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, layers)
        self.cls = nn.Linear(embed_dim, NUM_CLASSES)

    def forward(self, x, padding_mask=None):
        # Add positional encoding BEFORE transformer
        x = self.positional_encoding(x)
        h = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )
        return self.cls(h)


class SleepStagingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = EpochEncoder()
        self.context = SleepTransformer()

    def forward(self, x, padding_mask=None):
        feats = self.encoder(x)
        return self.context(feats, padding_mask)

In [9]:
class GatedFusion(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gate_net = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Sigmoid()
        )

    def forward(self, h_temp, h_spec):
        gate = self.gate_net(torch.cat([h_temp, h_spec], dim=-1))
        return gate * h_temp + (1 - gate) * h_spec


class BidirectionalCrossAttnFusion(nn.Module):
    """
    Temporal attends to spectral AND spectral attends to temporal.
    Outputs are combined via a learned gate.
    """
    def __init__(self, embed_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.temp2spec = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.spec2temp = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff_temp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim), nn.Dropout(dropout)
        )
        self.ff_spec = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2), nn.GELU(),
            nn.Linear(embed_dim * 2, embed_dim), nn.Dropout(dropout)
        )
        self.norm3 = nn.LayerNorm(embed_dim)
        self.norm4 = nn.LayerNorm(embed_dim)
        self.gate  = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.Sigmoid())

    def forward(self, h_temp, h_spec, key_padding_mask=None):
        # Temporal queries spectral
        a1, _ = self.temp2spec(h_temp, h_spec, h_spec, key_padding_mask=key_padding_mask)
        h_temp = self.norm1(h_temp + a1)
        h_temp = self.norm3(h_temp + self.ff_temp(h_temp))

        # Spectral queries temporal
        a2, _ = self.spec2temp(h_spec, h_temp, h_temp, key_padding_mask=key_padding_mask)
        h_spec = self.norm2(h_spec + a2)
        h_spec = self.norm4(h_spec + self.ff_spec(h_spec))

        # Learned gate to merge both enriched streams
        g = self.gate(torch.cat([h_temp, h_spec], dim=-1))
        return g * h_temp + (1 - g) * h_spec


class FusedSleepStagingModel(nn.Module):
    """
    Fuses temporal (CNN) and spectral (MLP) encoders before
    a shared SleepTransformer for sequence-level classification.

    Args:
        spectral_input_dim  — spectral feature vector size (default 34)
        embed_dim           — shared embedding dimension
        heads               — Transformer attention heads
        layers              — Transformer encoder layers
        dropout             — dropout rate
        fusion_type         — 'concat' | 'gated' | 'cross_attn'
    """
    def __init__(
        self,
        spectral_input_dim=34,
        embed_dim=128,
        heads=4,
        layers=6,             # increased from 3
        dropout=0.2,
        fusion_type='cross_attn'
    ):
        super().__init__()
        assert fusion_type in ('concat', 'gated', 'cross_attn')
        self.fusion_type = fusion_type

        # ── Temporal encoder (CNN pyramid) ──────────────────────────────────
        self.temporal_encoder = EpochEncoder(embed_dim=embed_dim)

        # ── Spectral encoder: MLP instead of single linear ──────────────────
        self.spectral_encoder = nn.Sequential(
            nn.Linear(spectral_input_dim, 128), nn.GELU(), nn.LayerNorm(128),
            nn.Linear(128, embed_dim),          nn.GELU(), nn.LayerNorm(embed_dim),
        )

        # ── Fusion module ────────────────────────────────────────────────────
        if fusion_type == 'concat':
            self.fusion = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.LayerNorm(embed_dim),
                nn.GELU()
            )
        elif fusion_type == 'gated':
            self.fusion = GatedFusion(embed_dim)
        else:  # cross_attn
            self.fusion = BidirectionalCrossAttnFusion(embed_dim, num_heads=heads, dropout=dropout)

        # ── Shared Transformer context model ─────────────────────────────────
        self.context = SleepTransformer(
            embed_dim=embed_dim,
            heads=heads,
            layers=layers,
            dropout=dropout
        )

        # ── Auxiliary per-stream heads (for auxiliary loss) ──────────────────
        self.temporal_cls = nn.Linear(embed_dim, NUM_CLASSES)
        self.spectral_cls = nn.Linear(embed_dim, NUM_CLASSES)

    def forward(self, x_temporal, x_spectral, padding_mask=None):
        """
        x_temporal:   [B, T, 3000]
        x_spectral:   [B, T, spectral_input_dim]
        padding_mask: [B, T]  — True for padded positions
        Returns:      logits [B, T, NUM_CLASSES],
                      aux_temp [B, T, NUM_CLASSES],
                      aux_spec [B, T, NUM_CLASSES]
        """
        h_temp = self.temporal_encoder(x_temporal)   # [B, T, D]
        h_spec = self.spectral_encoder(x_spectral)   # [B, T, D]

        # Auxiliary logits (before fusion, so each stream is forced to be discriminative)
        aux_temp = self.temporal_cls(h_temp)
        aux_spec = self.spectral_cls(h_spec)

        if self.fusion_type == 'concat':
            fused = self.fusion(torch.cat([h_temp, h_spec], dim=-1))
        elif self.fusion_type == 'gated':
            fused = self.fusion(h_temp, h_spec)
        else:  # cross_attn
            fused = self.fusion(h_temp, h_spec, key_padding_mask=padding_mask)

        logits = self.context(fused, padding_mask)   # [B, T, NUM_CLASSES]
        return logits, aux_temp, aux_spec

In [10]:
class FocalLoss(nn.Module):
    """
    Focal loss with:
      - padding mask (-100) correctly excluded from the mean
      - per-class alpha weights to handle class imbalance
    """
    def __init__(self, alpha=None, gamma=2.0, num_classes=5):
        super().__init__()
        self.gamma = gamma
        if alpha is None:
            self.register_buffer('alpha', torch.ones(num_classes))
        else:
            # alpha can be a tensor of per-class weights
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float))

    def forward(self, logits, targets):
        # logits:  [N, C]  (already flattened)
        # targets: [N]

        # --- FIXED: exclude padding tokens before averaging ---
        valid   = targets != -100
        logits  = logits[valid]
        targets = targets[valid]

        if logits.numel() == 0:
            return logits.sum() * 0.0   # safe zero grad

        ce   = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt   = torch.exp(-ce)
        loss = (1 - pt) ** self.gamma * ce
        return loss.mean()

In [11]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def compute_specificity(y_true, y_pred, num_classes=5):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    specs = []
    for i in range(num_classes):
        tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
        fp = cm[:, i].sum() - cm[i, i]
        specs.append(tn / (tn + fp + 1e-8))
    return float(np.mean(specs))


def compute_metrics(y_true, y_pred, y_prob):
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "precision": precision_score(y_true, y_pred, average="weighted"),
        "recall": recall_score(y_true, y_pred, average="weighted"),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "specificity": compute_specificity(y_true, y_pred)
    }
    print("[METRICS] Computed basic metrics (accuracy, F1, precision, recall, kappa, specificity).")
    try:
        metrics["roc_auc"] = roc_auc_score(
            y_true, y_prob, multi_class="ovr", average="weighted"
        )
        metrics["pr_auc"] = average_precision_score(
            y_true, y_prob, average="weighted"
        )
    except ValueError:
        metrics["roc_auc"] = None
        metrics["pr_auc"] = None

    return metrics

In [12]:
def logger(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

In [13]:
def evaluate_fusion(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    use_amp = device.type == 'cuda'

    with torch.no_grad():
        for x_temp, x_spec, y, padding_mask in loader:
            x_temp       = x_temp.to(device)
            x_spec       = x_spec.to(device)
            y            = y.to(device)
            padding_mask = padding_mask.to(device)

            with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                logits, _, _ = model(x_temp, x_spec, padding_mask)  # unpack 3 outputs

            probs = torch.softmax(logits, dim=-1)
            preds = probs.argmax(dim=-1)
            valid = ~padding_mask

            y_true.extend(y[valid].cpu().numpy())
            y_pred.extend(preds[valid].cpu().numpy())
            y_prob.extend(probs[valid].cpu().numpy())

    return compute_metrics(
        np.array(y_true),
        np.array(y_pred),
        np.array(y_prob)
    )

In [14]:
def train_model_fusion(train_dataset, val_dataset, device,
                       fusion_type='cross_attn', epochs=150):

    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=2,
        collate_fn=fusion_collate_fn,
        pin_memory=True,
        persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=16,
        shuffle=False,
        num_workers=2,
        collate_fn=fusion_collate_fn,
        pin_memory=True,
        persistent_workers=True
    )

    model = nn.DataParallel(
        FusedSleepStagingModel(fusion_type=fusion_type)
    ).to(device)

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger(f"Fused model on {device} | fusion='{fusion_type}'")
    logger(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")
    logger(f"Train chunks: {len(train_dataset):,} | Val chunks: {len(val_dataset):,}")

    # ── Per-class alpha: inverse-frequency weights ───────────────────────────
    class_counts  = torch.tensor([70154, 25175, 88983, 19454, 34184], dtype=torch.float)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES   # normalise to sum=5
    criterion = FocalLoss(alpha=class_weights, gamma=2.0).to(device)
    AUX_WEIGHT = 0.3   # weight for each auxiliary stream loss

    # ── AdamW + OneCycleLR ───────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-4,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.05      # 5% warmup
    )
    scaler  = torch.amp.GradScaler()
    best_f1 = -1

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch_idx, (x_temp, x_spec, y, padding_mask) in enumerate(train_loader):
            x_temp       = x_temp.to(device)
            x_spec       = x_spec.to(device)
            y            = y.to(device)
            padding_mask = padding_mask.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type='cuda'):
                logits, aux_temp, aux_spec = model(x_temp, x_spec, padding_mask)

                y_flat        = y.view(-1)
                loss_main     = criterion(logits.view(-1, NUM_CLASSES),    y_flat)
                loss_aux_temp = criterion(aux_temp.view(-1, NUM_CLASSES),  y_flat)
                loss_aux_spec = criterion(aux_spec.view(-1, NUM_CLASSES),  y_flat)

                loss = loss_main + AUX_WEIGHT * loss_aux_temp + AUX_WEIGHT * loss_aux_spec

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            if device.type == 'cuda' and batch_idx % 50 == 0:
                mem_alloc    = torch.cuda.memory_allocated() / 1024**2
                mem_reserved = torch.cuda.memory_reserved()  / 1024**2
                print(f"  GPU | Alloc: {mem_alloc:.1f} MB | Reserved: {mem_reserved:.1f} MB")

            total_loss += loss.item()

        metrics = evaluate_fusion(model, val_loader, device)

        log = {'epoch': epoch, 'train_loss': total_loss / len(train_loader), **metrics}
        with open('/kaggle/working/training_metrics_fusion.jsonl', 'a') as f:
            f.write(json.dumps(log) + '\n')

        if metrics['f1_weighted'] > best_f1:
            best_f1 = metrics['f1_weighted']
            torch.save(model.state_dict(), '/kaggle/working/best_model_fusion.pt')

        logger(
            f"Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f} | "
            f"F1: {metrics['f1_weighted']:.4f} | Acc: {metrics['accuracy']:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.6f}"
        )

    logger("Training complete — best model saved to best_model_fusion.pt")

In [15]:
def main():
    set_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    csv_path = '/kaggle/working/output.csv'

    # Split at FILE level (not chunk level) to prevent data leakage
    tmp = pd.read_csv(csv_path)
    num_files = len(tmp)
    print(f"Total files: {num_files}")

    indices = list(range(num_files))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

    WINDOW_SIZE = 256   # epochs per chunk — controls max sequence length
    OVERLAP     = 0     # set >0 (e.g. 64) for denser training signal

    train_dataset = FusionSleepDataset(csv_path, file_indices=train_idx,
                                       window_size=WINDOW_SIZE, overlap=OVERLAP)
    val_dataset   = FusionSleepDataset(csv_path, file_indices=val_idx,
                                       window_size=WINDOW_SIZE, overlap=OVERLAP)

    print(f"Train files: {len(train_idx)} → {len(train_dataset):,} chunks")
    print(f"Val   files: {len(val_idx)}  → {len(val_dataset):,} chunks")

    # Options: 'concat' | 'gated' | 'cross_attn'
    FUSION_TYPE = 'concat'

    train_model_fusion(train_dataset, val_dataset, device,
                       fusion_type=FUSION_TYPE, epochs=150)

In [16]:
# main()

In [17]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  REPLACE CELL 14 — inference + full evaluation                          ║
# ║  Includes: confusion matrix, classification report, ROC, PR curves      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
from pathlib import Path

FUSION_TYPE = 'concat'       # must match what was used in training
WINDOW_SIZE = 256
CLASS_NAMES = ['Wake', 'N1', 'N2', 'N3', 'REM']
NUM_CLASSES = 5
COLORS      = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))
OUTPUT_DIR  = Path('/kaggle/working/evaluation_results')
OUTPUT_DIR.mkdir(exist_ok=True)

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
csv_path = '/kaggle/working/output.csv'

# ── Load dataset & model ────────────────────────────────────────────────────
test_dataset = FusionSleepDataset(csv_path, window_size=WINDOW_SIZE, overlap=0)
test_loader  = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    collate_fn=fusion_collate_fn,
    pin_memory=True,
    persistent_workers=True
)
print(f"Full dataset: {len(test_dataset):,} chunks from {len(test_dataset.df)} files")

model = nn.DataParallel(FusedSleepStagingModel(fusion_type=FUSION_TYPE)).to(device)
model.load_state_dict(
    torch.load('/kaggle/working/best_model_fusion.pt', map_location=device)
)
model.eval()
print("Best fusion model loaded!\n")

# ── Collect predictions ─────────────────────────────────────────────────────
print("Collecting predictions...")
y_true_all, y_pred_all, y_prob_all = [], [], []

with torch.no_grad():
    for batch_idx, (x_temp, x_spec, y, padding_mask) in enumerate(test_loader):
        x_temp, x_spec = x_temp.to(device), x_spec.to(device)
        y, padding_mask = y.to(device), padding_mask.to(device)

        with torch.amp.autocast(device_type='cuda', enabled=device.type == 'cuda'):
            logits = model(x_temp, x_spec, padding_mask)[0]

        probs = torch.softmax(logits, dim=-1)
        preds = probs.argmax(dim=-1)
        valid = ~padding_mask

        y_true_all.extend(y[valid].cpu().numpy())
        y_pred_all.extend(preds[valid].cpu().numpy())
        y_prob_all.extend(probs[valid].cpu().numpy())

        if batch_idx % 20 == 0:
            print(f"  Batch {batch_idx}/{len(test_loader)}")

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_prob_all = np.array(y_prob_all)
y_true_bin = label_binarize(y_true_all, classes=list(range(NUM_CLASSES)))  # [N, 5]
print(f"Predictions collected: {len(y_true_all):,} valid epochs\n")

# ── [1] Classification Report ───────────────────────────────────────────────
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(y_true_all, y_pred_all, target_names=CLASS_NAMES, digits=4)
print(report)
with open(OUTPUT_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)

# ── [2] Confusion Matrix ────────────────────────────────────────────────────
cm      = confusion_matrix(y_true_all, y_pred_all)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm,      annot=True, fmt='d',    cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title(f'Confusion Matrix — Counts [{FUSION_TYPE} fusion]', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title(f'Confusion Matrix — Normalised [{FUSION_TYPE} fusion]', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved confusion matrix")

# ── [3] ROC Curves (one-vs-rest per class + micro-average) ─────────────────
fpr, tpr, roc_auc = {}, {}, {}

for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob_all[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Micro-average: flatten all classes together
fpr['micro'], tpr['micro'], _ = roc_curve(y_true_bin.ravel(), y_prob_all.ravel())
roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])

# Macro-average: interpolate each class ROC onto a common grid then average
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= NUM_CLASSES
fpr['macro'], tpr['macro'] = all_fpr, mean_tpr
roc_auc['macro'] = auc(fpr['macro'], tpr['macro'])

fig, ax = plt.subplots(figsize=(10, 8))

for i, color in zip(range(NUM_CLASSES), COLORS):
    ax.plot(fpr[i], tpr[i], color=color, lw=2,
            label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})')

ax.plot(fpr['micro'], tpr['micro'], color='deeppink',
        linestyle='--', lw=2.5,
        label=f'Micro-average (AUC = {roc_auc["micro"]:.3f})')
ax.plot(fpr['macro'], tpr['macro'], color='navy',
        linestyle=':', lw=2.5,
        label=f'Macro-average (AUC = {roc_auc["macro"]:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(f'ROC Curves — One-vs-Rest [{FUSION_TYPE} fusion]',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved ROC curves  |  Macro-AUROC: {roc_auc['macro']:.4f}  |  Micro-AUROC: {roc_auc['micro']:.4f}")

# ── [4] Precision-Recall Curves (per class + micro-average) ────────────────
precision_dict, recall_dict, ap_dict = {}, {}, {}

for i in range(NUM_CLASSES):
    precision_dict[i], recall_dict[i], _ = precision_recall_curve(
        y_true_bin[:, i], y_prob_all[:, i]
    )
    ap_dict[i] = average_precision_score(y_true_bin[:, i], y_prob_all[:, i])

# Micro-average PR
precision_dict['micro'], recall_dict['micro'], _ = precision_recall_curve(
    y_true_bin.ravel(), y_prob_all.ravel()
)
ap_dict['micro'] = average_precision_score(y_true_bin, y_prob_all, average='micro')

# Macro-average AP (mean of per-class APs)
ap_dict['macro'] = average_precision_score(y_true_bin, y_prob_all, average='macro')

fig, ax = plt.subplots(figsize=(10, 8))

for i, color in zip(range(NUM_CLASSES), COLORS):
    ax.plot(recall_dict[i], precision_dict[i], color=color, lw=2,
            label=f'{CLASS_NAMES[i]} (AP = {ap_dict[i]:.3f})')

ax.plot(recall_dict['micro'], precision_dict['micro'],
        color='deeppink', linestyle='--', lw=2.5,
        label=f'Micro-average (AP = {ap_dict["micro"]:.3f})')

# Baseline: random classifier AP = class prevalence
baseline = y_true_bin.sum(axis=0) / len(y_true_all)
for i, color in zip(range(NUM_CLASSES), COLORS):
    ax.axhline(baseline[i], color=color, linestyle=':', lw=0.8, alpha=0.5)

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f'Precision-Recall Curves [{FUSION_TYPE} fusion]',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pr_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved PR curves   |  Macro-AP: {ap_dict['macro']:.4f}  |  Micro-AP: {ap_dict['micro']:.4f}")

# ── [5] Summary table ───────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("PER-CLASS AUROC & AVERAGE PRECISION SUMMARY")
print("=" * 60)
print(f"{'Class':<10} {'AUROC':>8} {'Avg Precision':>15}")
print("-" * 35)
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<10} {roc_auc[i]:>8.4f} {ap_dict[i]:>15.4f}")
print("-" * 35)
print(f"{'Macro':<10} {roc_auc['macro']:>8.4f} {ap_dict['macro']:>15.4f}")
print(f"{'Micro':<10} {roc_auc['micro']:>8.4f} {ap_dict['micro']:>15.4f}")
print("=" * 60)
print(f"\n✓ All results saved to: {OUTPUT_DIR.absolute()}")

Full dataset: 1,023 chunks from 197 files


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/best_model_fusion.pt'

In [ ]:
# Use .value instead of .data
if uploader.value:
    # uploader.value is a list; [0] gets the first file
    # ['content'] contains the actual bytes
    file_content = uploader.value[0]['content']
    
    with open("best_model_fusion.pt", "wb") as f:
        f.write(file_content)
    
    print("Model uploaded successfully to /kaggle/working/best_model.pt")
else:
    print("No file uploaded yet! Please select a file before running this cell.")


In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create and display the upload button
uploader = widgets.FileUpload(accept='.pt', multiple=False)
display(uploader)

In [ ]:
csv_path = '/kaggle/working/output.csv'

# Split at FILE level (not chunk level) to prevent data leakage
tmp = pd.read_csv(csv_path)
num_files = len(tmp)
print(f"Total files: {num_files}")

indices = list(range(num_files))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

WINDOW_SIZE = 256   # epochs per chunk — controls max sequence length
OVERLAP     = 0     # set >0 (e.g. 64) for denser training signal

train_dataset = FusionSleepDataset(csv_path, file_indices=train_idx,
                                   window_size=WINDOW_SIZE, overlap=OVERLAP)
val_dataset   = FusionSleepDataset(csv_path, file_indices=val_idx,
                                   window_size=WINDOW_SIZE, overlap=OVERLAP)
val_loader = DataLoader(
        val_dataset,
        batch_size=16,
        shuffle=False,
        num_workers=2,
        collate_fn=fusion_collate_fn,
        pin_memory=True,
        persistent_workers=True
    )

In [ ]:
class AttentionRollout:
    """
    Extracts per-epoch influence scores by accumulating attention weights
    across transformer layers via monkey-patching to force need_weights=True,
    bypassing PyTorch's fast path which returns None for weights.
    """
    def __init__(self, model):
        self.model     = model
        self.attn_maps = []
        self._patched  = []

    def _patch(self):
        """Temporarily replace each self_attn.forward to capture weights."""
        self.attn_maps.clear()
        self._patched.clear()
        captured = self.attn_maps

        for layer in self.model.module.context.encoder.layers:
            mha  = layer.self_attn
            orig = mha.forward

            def make_patched(original):
                def patched(query, key, value, key_padding_mask=None,
                            need_weights=True, attn_mask=None, **kwargs):
                    out, weights = original(
                        query, key, value,
                        key_padding_mask=key_padding_mask,
                        need_weights=True,          # force — bypasses fast path
                        average_attn_weights=True,
                        attn_mask=attn_mask
                    )
                    captured.append(weights.detach() if weights is not None else None)
                    return out, weights
                return patched

            mha.forward = make_patched(orig)
            self._patched.append((mha, orig))

    def _unpatch(self):
        for mha, orig in self._patched:
            mha.forward = orig
        self._patched.clear()

    def __call__(self, x_temp, x_spec, padding_mask=None):
        self._patch()
        self.model.eval()

        with torch.no_grad():
            logits, _, _ = self.model(x_temp, x_spec, padding_mask)

        self._unpatch()

        valid_maps = [a for a in self.attn_maps if a is not None]

        # Fallback: if all layers returned None, return uniform influence
        if not valid_maps:
            B, T = x_temp.shape[:2]
            influence = torch.ones(B, T, device=x_temp.device) / T
            if padding_mask is not None:
                influence[padding_mask] = 0.0
            return influence, torch.softmax(logits, dim=-1)

        # Attention rollout: accumulate with residual (0.5*A + 0.5*I)
        rollout = None
        for attn in valid_maps:
            # attn: [B, T, T]
            I      = torch.eye(attn.size(-1), device=attn.device).unsqueeze(0)
            attn_r = 0.5 * attn + 0.5 * I
            attn_r = attn_r / attn_r.sum(dim=-1, keepdim=True)
            rollout = attn_r if rollout is None else torch.bmm(rollout, attn_r)

        influence = rollout.mean(dim=1)  # [B, T]

        if padding_mask is not None:
            influence[padding_mask] = 0.0

        return influence, torch.softmax(logits, dim=-1)

In [ ]:
class EpochGradCAM:
    """
    GradCAM on the AdaptiveAtrousPyramid proj layer.
    Uses a single forward pass with retain_grad() so the gradient
    flows correctly back to the captured feature map.
    """
    def __init__(self, model):
        self.model = model
        self.fmap  = None

    def compute(self, x_temp, x_spec, padding_mask, epoch_idx, batch_idx=0):
        """
        epoch_idx : which epoch (time step) in the window to explain
        Returns   : heatmap [3000] normalised to [0, 1]
        """
        self.model.eval()
        self.fmap = None

        encoder = self.model.module.temporal_encoder
        pyramid = encoder.pyramid

        # Hook captures the proj layer output during the SINGLE full forward
        def fwd_hook(m, inp, out):
            self.fmap = out          # [B*T, 64, T']
            self.fmap.retain_grad()  # needed to access .grad after backward

        handle = pyramid.proj.register_forward_hook(fwd_hook)

        # ONE forward pass — gradients flow to self.fmap correctly
        with torch.enable_grad():
            logits, _, _ = self.model(x_temp, x_spec, padding_mask)
            class_score  = logits[batch_idx, epoch_idx].max()
            class_score.backward()

        handle.remove()

        if self.fmap is None or self.fmap.grad is None:
            print("[GradCAM] Warning: no gradient captured — returning zeros")
            return np.zeros(3000)

        # x_temp is [1, T, 3000] (single sample), so flat index = epoch_idx
        T        = x_temp.shape[1]
        flat_idx = batch_idx * T + epoch_idx

        fmap_ep = self.fmap[flat_idx]       # [64, T']
        grad_ep = self.fmap.grad[flat_idx]  # [64, T']

        weights = grad_ep.mean(dim=-1)      # [64]
        cam     = (weights.unsqueeze(-1) * fmap_ep).sum(dim=0)  # [T']
        cam     = F.relu(cam)

        # Upsample from T' → 3000 raw timepoints
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0), size=3000,
            mode='linear', align_corners=False
        ).squeeze()

        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy()

In [ ]:
class MCDropoutPredictor:
    """
    Monte Carlo Dropout uncertainty estimation.
    Requires dropout > 0 in the model (already set to 0.2).
    """
    def __init__(self, model, n_samples=30):
        self.model     = model
        self.n_samples = n_samples

    def _enable_dropout(self):
        """Turn on dropout even during eval mode."""
        for m in self.model.modules():
            if isinstance(m, nn.Dropout):
                m.train()

    def predict(self, x_temp, x_spec, padding_mask=None):
        """
        Returns:
            mean_probs  : [B, T, C]  — mean predicted probabilities
            uncertainty : [B, T]     — predictive entropy (higher = less certain)
            pred_std    : [B, T, C]  — std across MC samples per class
        """
        self.model.eval()
        self._enable_dropout()   # dropout ON during inference

        all_probs = []
        with torch.no_grad():
            for _ in range(self.n_samples):
                logits, _, _ = self.model(x_temp, x_spec, padding_mask)
                all_probs.append(torch.softmax(logits, dim=-1))

        all_probs   = torch.stack(all_probs, dim=0)  # [N_samples, B, T, C]
        mean_probs  = all_probs.mean(dim=0)           # [B, T, C]
        pred_std    = all_probs.std(dim=0)            # [B, T, C]

        # Predictive entropy: H = -sum(p * log(p))
        eps         = 1e-8
        entropy     = -(mean_probs * (mean_probs + eps).log()).sum(dim=-1)  # [B, T]

        # Zero out padding
        if padding_mask is not None:
            entropy[padding_mask]    = 0.0
            mean_probs[padding_mask] = 0.0

        return mean_probs, entropy, pred_std


# # Usage
# mc = MCDropoutPredictor(model, n_samples=30)
# mean_probs, uncertainty, pred_std = mc.predict(x_temp, x_spec, padding_mask)
# uncertainty: [B, T] — entropy per epoch, high value = model unsure
# pred_std:    [B, T, 5] — spread per class across samples

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.ticker import MaxNLocator
from scipy.ndimage import uniform_filter1d
import numpy as np

# ── Clinical display settings ────────────────────────────────────────────────
STAGE_NAMES  = ['Wake', 'N1', 'N2', 'N3', 'REM']
STAGE_COLORS = {
    0: '#E05C5C',   # Wake  — warm red
    1: '#A78BCA',   # N1    — soft purple
    2: '#5B9BD5',   # N2    — calm blue
    3: '#4BAE8A',   # N3    — deep green
    4: '#F0A500',   # REM   — amber
}
MAX_EPOCHS = 180

# Confidence thresholds (entropy-based, hidden from clinician)
# These map uncertainty → plain language
def _confidence_label(entropy_val, max_entropy):
    ratio = entropy_val / (max_entropy + 1e-8)
    if ratio < 0.45:
        return 'High',   '#2ecc71', '●'
    elif ratio < 0.72:
        return 'Medium', '#f39c12', '●'
    else:
        return 'Low',    '#e74c3c', '●'


def explain_single_sample(model, x_temp, x_spec, padding_mask, y_true,
                           batch_idx=0, epoch_to_detail=None):

    device = next(model.parameters()).device

    # ── Extract single sample ────────────────────────────────────────────────
    x_t   = x_temp[batch_idx].unsqueeze(0).to(device)
    x_s   = x_spec[batch_idx].unsqueeze(0).to(device)
    pmask = padding_mask[batch_idx].unsqueeze(0).to(device)
    y_t   = y_true[batch_idx]

    T_full = (~pmask[0]).sum().item()
    T      = min(T_full, MAX_EPOCHS)

    # ── Run explainers (all technical, none shown to clinician) ─────────────
    torch.cuda.empty_cache()
    rollout = AttentionRollout(model)
    influence, probs = rollout(x_t, x_s, pmask)
    influence_ = influence[0, :T].cpu().numpy()
    preds      = probs[0, :T].argmax(dim=-1).cpu().numpy()
    del rollout, probs
    torch.cuda.empty_cache()

    mc = MCDropoutPredictor(model, n_samples=20)
    mean_probs, entropy, _ = mc.predict(x_t, x_s, pmask)
    entropy_    = entropy[0, :T].cpu().numpy()
    del mc, mean_probs, entropy
    torch.cuda.empty_cache()

    if epoch_to_detail is None:
        # Pick most uncertain epoch where model was also wrong
        wrong_mask = (preds != y_t[:T].cpu().numpy())
        if wrong_mask.any():
            epoch_to_detail = int(np.argmax(entropy_ * wrong_mask))
        else:
            epoch_to_detail = int(np.argmax(entropy_))
    epoch_to_detail = min(epoch_to_detail, T - 1)

    gradcam = EpochGradCAM(model)
    heatmap = gradcam.compute(x_t, x_s, pmask,
                              epoch_idx=epoch_to_detail, batch_idx=0)
    del gradcam
    torch.cuda.empty_cache()

    y_true_ = y_t[:T].cpu().numpy().astype(int)
    preds_  = preds.astype(int)
    epochs  = np.arange(T)
    time_minutes = epochs * 0.5   # each epoch = 30 s → 0.5 min

    # Derive confidence level per epoch
    max_ent = np.log(5)   # max possible entropy for 5 classes
    conf_levels = [_confidence_label(entropy_[i], max_ent) for i in range(T)]

    # Identify epochs flagged for review: low confidence OR wrong prediction
    flagged = np.where(
        (preds_ != y_true_) | (entropy_ > np.percentile(entropy_, 75))
    )[0]

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 1 — Sleep Stage Summary (the Hypnogram a clinician knows)
    # ════════════════════════════════════════════════════════════════════════
    max_ent      = np.log(5)                          # max entropy for 5 classes
    norm_entropy = np.clip(entropy_ / max_ent, 0, 1)  # 0=certain, 1=most uncertain
    confidence   = 1.0 - norm_entropy                 # 0=low conf, 1=high conf

    # ── 5 bins, jet colourmap ────────────────────────────────────────────────
    N_BINS   = 5
    jet      = cm.get_cmap('jet', N_BINS)
    bounds   = np.linspace(0, 1, N_BINS + 1)          # [0, 0.2, 0.4, 0.6, 0.8, 1.0]
    bin_norm = mcolors.BoundaryNorm(bounds, N_BINS)
    bin_idx  = np.digitize(confidence, bounds) - 1
    bin_idx  = np.clip(bin_idx, 0, N_BINS - 1)

    fig1, axes1 = plt.subplots(
        2, 1, figsize=(18, 7),
        gridspec_kw={'height_ratios': [4, 1.2], 'hspace': 0.08}
    )
    fig1.patch.set_facecolor('#F7F9FC')

    ax_hyp  = axes1[0]
    ax_disc = axes1[1]
    ax_hyp.set_facecolor('#F7F9FC')
    ax_disc.set_facecolor('#F7F9FC')

    # ── Hypnogram: confidence-coloured epoch background strips ───────────────
    for i in range(T):
        color = jet(bin_idx[i] / (N_BINS - 1))
        ax_hyp.axvspan(
            time_minutes[i] - 0.25,
            time_minutes[i] + 0.25,
            color=color, alpha=0.45, linewidth=0, zorder=1
        )

    # ── Ground truth — solid thick GREEN ────────────────────────────────────
    GT_COLOR   = '#1a7a1a'
    PRED_COLOR = '#cc1111'

    for i in range(T - 1):
        ax_hyp.plot(
            [time_minutes[i], time_minutes[i + 1]],
            [y_true_[i], y_true_[i]],
            color=GT_COLOR, lw=4.5, solid_capstyle='butt', zorder=3
        )
        ax_hyp.plot(
            [time_minutes[i + 1], time_minutes[i + 1]],
            [y_true_[i], y_true_[i + 1]],
            color=GT_COLOR, lw=4.5, zorder=3
        )

    # ── AI prediction — solid thick RED ─────────────────────────────────────
    for i in range(T - 1):
        ax_hyp.plot(
            [time_minutes[i], time_minutes[i + 1]],
            [preds_[i], preds_[i]],
            color=PRED_COLOR, lw=2.0, linestyle='--', zorder=4
        )
        ax_hyp.plot(
            [time_minutes[i + 1], time_minutes[i + 1]],
            [preds_[i], preds_[i + 1]],
            color=PRED_COLOR, lw=2.0, linestyle='--', zorder=4
        )

    # ── Mark the explained epoch ─────────────────────────────────────────────
    ax_hyp.axvline(time_minutes[epoch_to_detail], color='#2c2c2c',
                   lw=1.8, linestyle=':', zorder=5)
    ax_hyp.text(
        time_minutes[epoch_to_detail] + 0.3, 4.65,
        f'▼ Reviewed below\n({time_minutes[epoch_to_detail]:.0f} min mark)',
        fontsize=8.5, color='#2c2c2c', va='top', fontweight='bold'
    )

    ax_hyp.set_yticks(range(5))
    ax_hyp.set_yticklabels(STAGE_NAMES, fontsize=12, fontweight='bold')
    ax_hyp.set_xlim(-0.5, time_minutes[-1] + 0.5)
    ax_hyp.set_ylim(-0.6, 5.4)
    ax_hyp.set_xticks([])
    ax_hyp.spines[['top', 'right', 'bottom']].set_visible(False)
    ax_hyp.set_ylabel('Sleep Stage', fontsize=11)

    # ── Agreement stat ───────────────────────────────────────────────────────
    n_correct = (preds_ == y_true_).sum()
    pct       = 100 * n_correct / T
    pct_color = '#27ae60' if pct >= 80 else '#e67e22' if pct >= 65 else '#e74c3c'
    ax_hyp.text(
        0.01, 0.97,
        f'AI Agreement with Recording:  {pct:.0f}%  ({n_correct}/{T} epochs)',
        transform=ax_hyp.transAxes, fontsize=10,
        color=pct_color, fontweight='bold', va='top'
    )

    # ── Legend: lines ────────────────────────────────────────────────────────
    gt_line   = plt.Line2D([0], [0], color=GT_COLOR,   lw=4.5,
                            label='Recorded (PSG)')
    pred_line = plt.Line2D([0], [0], color=PRED_COLOR, lw=2.0,
                            linestyle='--', label='AI Assessment')
    line_legend = ax_hyp.legend(
        handles=[gt_line, pred_line],
        loc='upper right', fontsize=9.5, framealpha=0.95,
        title='Sleep stage lines'
    )
    ax_hyp.add_artist(line_legend)

    # ── Legend: confidence colourbar (jet, Low → High) ───────────────────────
    sm_conf  = cm.ScalarMappable(
        cmap=jet,
        norm=mcolors.Normalize(vmin=0, vmax=1)
    )
    sm_conf.set_array([])
    cbar_ax  = fig1.add_axes([0.92, 0.38, 0.012, 0.52])   # [left, bottom, w, h]
    cbar     = fig1.colorbar(sm_conf, cax=cbar_ax,
                              boundaries=bounds,
                              ticks=[0, 0.25, 0.5, 0.75, 1.0])
    cbar.set_ticklabels(['Low\n(0.0)', '0.25', '0.50', '0.75', 'High\n(1.0)'],
                        fontsize=8)
    cbar.set_label('AI Confidence', fontsize=9, labelpad=6)
    cbar.ax.set_title('confidence', fontsize=7.5, pad=4)

    # ── Strip 2: Correct (green) / Wrong (red) ───────────────────────────────
    for i in range(T):
        match = (preds_[i] == y_true_[i])
        ax_disc.bar(
            time_minutes[i], 1, width=0.47,
            color='#2ecc71' if match else '#e74c3c',
            alpha=0.9
        )

    ax_disc.set_xlim(-0.5, time_minutes[-1] + 0.5)
    ax_disc.set_ylim(0, 1)
    ax_disc.set_yticks([])
    ax_disc.set_ylabel('')                             # no y-axis title
    ax_disc.set_xlabel('Time into recording (minutes)', fontsize=11)
    ax_disc.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

    # ── Legend: correct vs wrong ─────────────────────────────────────────────
    agree_p   = mpatches.Patch(color='#2ecc71', label='Correct prediction')
    wrong_p   = mpatches.Patch(color='#e74c3c', label='Wrong prediction')
    ax_disc.legend(
        handles=[agree_p, wrong_p],
        loc='upper right', fontsize=9, framealpha=0.95,
        ncol=2
    )

    fig1.subplots_adjust(right=0.90)    # make room for the colourbar
    fig1.savefig('clinical_fig1_sleep_summary.png', dpi=150, bbox_inches='tight',
                 facecolor='#F7F9FC')
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 2 — Epochs Flagged for Clinical Review
    # ════════════════════════════════════════════════════════════════════════
    fig2, ax2 = plt.subplots(figsize=(18, 6))
    fig2.patch.set_facecolor('#F7F9FC')
    ax2.set_facecolor('#F7F9FC')

    # Draw each epoch as a coloured block on a timeline
    for i in range(T):
        label, conf_color, _ = conf_levels[i]
        match = (preds_[i] == y_true_[i])

        # Block height = 1, coloured by stage
        ax2.bar(time_minutes[i], 0.6, width=0.47, bottom=0.2,
                color=STAGE_COLORS[y_true_[i]], alpha=0.6, zorder=2)

        # Confidence dot above each block
        dot_color = conf_color
        ax2.scatter(time_minutes[i], 1.05, s=28,
                    color=dot_color, zorder=4, marker='o')

        # Red border if disagreement
        if not match:
            ax2.bar(time_minutes[i], 0.6, width=0.47, bottom=0.2,
                    fill=False, edgecolor='#e74c3c', linewidth=1.5, zorder=3)

    # Annotate only the worst disagreements (low conf + wrong)
    low_conf_wrong = [
        i for i in range(T)
        if preds_[i] != y_true_[i] and conf_levels[i][0] == 'Low'
    ]
    # Show top 5 worst
    top_bad = sorted(low_conf_wrong,
                     key=lambda i: entropy_[i], reverse=True)[:5]
    for rank, i in enumerate(top_bad):
        ax2.annotate(
            f"Review\n{time_minutes[i]:.0f} min\n"
            f"AI said: {STAGE_NAMES[preds_[i]]}\n"
            f"Record: {STAGE_NAMES[y_true_[i]]}",
            xy=(time_minutes[i], 0.85),
            xytext=(time_minutes[i], 1.45 + (rank % 2) * 0.55),
            fontsize=7.5, ha='center', color='#c0392b', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1.0),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#fdecea',
                      edgecolor='#e74c3c', alpha=0.9)
        )

    ax2.set_xlim(-0.5, time_minutes[-1] + 0.5)
    ax2.set_ylim(0, 2.6)
    ax2.set_yticks([])
    ax2.set_xlabel('Time into recording (minutes)', fontsize=11)
    ax2.spines[['top', 'right', 'left', 'bottom']].set_visible(False)

    # Legend
    stage_patches = [mpatches.Patch(color=STAGE_COLORS[i],
                                    alpha=0.7, label=STAGE_NAMES[i])
                     for i in range(5)]
    conf_high  = plt.Line2D([0], [0], marker='o', color='w',
                             markerfacecolor='#2ecc71', markersize=9,
                             label='High confidence')
    conf_med   = plt.Line2D([0], [0], marker='o', color='w',
                             markerfacecolor='#f39c12', markersize=9,
                             label='Medium confidence')
    conf_low   = plt.Line2D([0], [0], marker='o', color='w',
                             markerfacecolor='#e74c3c', markersize=9,
                             label='Low confidence')
    border_p   = mpatches.Patch(fill=False, edgecolor='#e74c3c',
                                 linewidth=1.5, label='AI disagrees with recording')
    ax2.legend(
        handles=[conf_high, conf_med, conf_low, border_p] + stage_patches,
        loc='upper left', fontsize=8.5, framealpha=0.95, ncol=4,
        title='Block colour = recorded stage   |   Dot = AI confidence   |   Red border = disagreement'
    )

    n_flag = len(top_bad)
    ax2.set_title(
        f'Epochs Recommended for Manual Review\n'
        f'{n_flag} epoch{"s" if n_flag != 1 else ""} flagged where the AI was '
        f'both uncertain and disagreed with the recording — shown with labels above',
        fontsize=13, fontweight='bold', loc='left', pad=10
    )
    fig2.savefig('clinical_fig2_review_flags.png', dpi=150, bbox_inches='tight',
                 facecolor='#F7F9FC')
    plt.show()

    # ════════════════════════════════════════════════════════════════════════
    # FIGURE 3 — Signal Evidence for the Flagged Epoch
    # ════════════════════════════════════════════════════════════════════════
    fig3 = plt.figure(figsize=(18, 7))
    fig3.patch.set_facecolor('#F7F9FC')
    gs3  = gridspec.GridSpec(
        2, 2, figure=fig3,
        width_ratios=[3.5, 1],
        height_ratios=[1, 4],
        hspace=0.06, wspace=0.25
    )

    ax_label  = fig3.add_subplot(gs3[0, 0])   # Header strip
    ax_signal = fig3.add_subplot(gs3[1, 0])   # EEG signal
    ax_card   = fig3.add_subplot(gs3[:, 1])   # Summary card

    for ax in [ax_label, ax_signal, ax_card]:
        ax.set_facecolor('#F7F9FC')

    time_axis  = np.linspace(0, 30, 3000)
    raw_signal = x_t[0, epoch_to_detail].cpu().numpy()
    pred_stage = STAGE_NAMES[preds_[epoch_to_detail]]
    true_stage = STAGE_NAMES[y_true_[epoch_to_detail]]
    correct    = (pred_stage == true_stage)
    conf_lbl, conf_color, _ = conf_levels[epoch_to_detail]

    # ── Header strip: shows where this epoch sits in the night ──────────────
    for i in range(T):
        ax_label.bar(time_minutes[i], 1, width=0.47,
                     color=STAGE_COLORS[y_true_[i]], alpha=0.5)
    ax_label.axvline(time_minutes[epoch_to_detail], color='#c0392b',
                     lw=3, zorder=5)
    ax_label.set_xlim(-0.5, time_minutes[-1] + 0.5)
    ax_label.set_ylim(0, 1)
    ax_label.set_yticks([])
    ax_label.set_xticks([])
    ax_label.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    ax_label.text(
        time_minutes[epoch_to_detail], 1.08,
        f'▼ This epoch ({time_minutes[epoch_to_detail]:.0f} min)',
        ha='center', va='bottom', fontsize=9,
        color='#c0392b', fontweight='bold',
        transform=ax_label.get_xaxis_transform()
    )
    ax_label.set_title(
        f'30-Second EEG Signal at the {time_minutes[epoch_to_detail]:.0f}-Minute Mark  '
        f'(Epoch {epoch_to_detail} of {T})',
        fontsize=12, fontweight='bold', loc='left', pad=4
    )

    # ── EEG signal with highlighted regions ─────────────────────────────────
    # Smooth heatmap to find meaningful focus regions
    smooth_hm = uniform_filter1d(heatmap, size=200)
    threshold = np.percentile(smooth_hm, 80)

    # Shade low-attention (background) green, high-attention red
    ax_signal.fill_between(
        time_axis, raw_signal.min() * 1.4, raw_signal.max() * 1.4,
        color='#d5f5e3', alpha=0.5, label='_nolegend_'
    )
    # Highlight high-attention regions
    in_region = False
    region_start = 0
    region_count = 0
    for i, v in enumerate(smooth_hm):
        if v >= threshold and not in_region:
            region_start = i
            in_region    = True
        elif v < threshold and in_region:
            region_count += 1
            ax_signal.fill_between(
                time_axis[region_start:i],
                raw_signal.min() * 1.4,
                raw_signal.max() * 1.4,
                color='#fadbd8', alpha=0.75,
                label='Wave pattern the AI focused on' if region_count == 1 else '_nolegend_'
            )
            in_region = False
    if in_region:
        ax_signal.fill_between(
            time_axis[region_start:],
            raw_signal.min() * 1.4,
            raw_signal.max() * 1.4,
            color='#fadbd8', alpha=0.75
        )

    ax_signal.plot(time_axis, raw_signal, color='#1a1a2e', lw=1.0, zorder=4)

    # Annotate focus regions plainly
    focus_regions = []
    in_region = False
    for i, v in enumerate(smooth_hm):
        if v >= threshold and not in_region:
            region_start = i; in_region = True
        elif v < threshold and in_region:
            focus_regions.append((region_start, i)); in_region = False
    focus_regions = sorted(focus_regions,
                           key=lambda r: smooth_hm[r[0]:r[1]].mean(),
                           reverse=True)[:3]
    for j, (rs, re) in enumerate(focus_regions):
        t_mid = time_axis[(rs + re) // 2]
        y_pos = raw_signal.max() * 1.15
        ax_signal.annotate(
            f'Pattern {j+1}\n(AI focused here)',
            xy=(t_mid, raw_signal[(rs + re) // 2]),
            xytext=(t_mid, y_pos),
            fontsize=8, ha='center', color='#922b21', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#922b21', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.25', facecolor='#fdecea',
                      edgecolor='#e74c3c', alpha=0.9)
        )

    ax_signal.set_xlim(0, 30)
    ax_signal.set_ylim(raw_signal.min() * 1.45, raw_signal.max() * 1.55)
    ax_signal.set_xlabel('Time within this 30-second window (seconds)', fontsize=11)
    ax_signal.set_ylabel('EEG Signal\nAmplitude', fontsize=10)
    ax_signal.spines[['top', 'right']].set_visible(False)

    green_p = mpatches.Patch(color='#d5f5e3', alpha=0.8,
                              label='Background signal (not flagged)')
    red_p   = mpatches.Patch(color='#fadbd8', alpha=0.9,
                              label='Wave patterns the AI focused on')
    ax_signal.legend(handles=[green_p, red_p], loc='lower right',
                     fontsize=9, framealpha=0.9)

    # ── Summary card ─────────────────────────────────────────────────────────
    ax_card.axis('off')

    card_bg = '#fff8f0' if not correct else '#f0fff4'
    card_edge = '#e74c3c' if not correct else '#27ae60'
    fig3.patches.append(
        mpatches.FancyBboxPatch(
            (0.02, 0.02), 0.96, 0.96,
            boxstyle='round,pad=0.04',
            transform=ax_card.transAxes,
            facecolor=card_bg,
            edgecolor=card_edge,
            linewidth=2.5,
            zorder=0
        )
    )

    verdict_text = 'MATCHES RECORDING' if correct else 'DOES NOT MATCH'
    verdict_col  = '#1e8449' if correct else '#c0392b'
    verdict_icon = '✓' if correct else '✗'

    lines = [
        (0.5, 0.93, f'{verdict_icon}  AI Assessment', 16, verdict_col, 'bold', 'center'),
        (0.5, 0.84, verdict_text,                      13, verdict_col, 'bold', 'center'),
        (0.5, 0.74, '─' * 22,                          10, '#aaaaaa', 'normal', 'center'),
        (0.08, 0.65, 'Recording shows:',                9,  '#555555', 'normal', 'left'),
        (0.92, 0.65, true_stage,                        12, STAGE_COLORS[y_true_[epoch_to_detail]],
         'bold', 'right'),
        (0.08, 0.54, 'AI assessed as:',                 9,  '#555555', 'normal', 'left'),
        (0.92, 0.54, pred_stage,                        12, STAGE_COLORS[preds_[epoch_to_detail]],
         'bold', 'right'),
        (0.5, 0.44, '─' * 22,                          10, '#aaaaaa', 'normal', 'center'),
        (0.08, 0.35, 'AI Confidence:',                   9,  '#555555', 'normal', 'left'),
        (0.92, 0.35, conf_lbl,                          12, conf_color, 'bold', 'right'),
        (0.08, 0.25, 'Time in recording:',               9,  '#555555', 'normal', 'left'),
        (0.92, 0.25, f'{time_minutes[epoch_to_detail]:.0f} min', 11, '#333333', 'bold', 'right'),
        (0.08, 0.15, 'Epoch number:',                    9,  '#555555', 'normal', 'left'),
        (0.92, 0.15, f'{epoch_to_detail} / {T}',        10, '#333333', 'bold', 'right'),
    ]
    if not correct:
        lines.append(
            (0.5, 0.05,
             '⚠  Manual review recommended',
             8.5, '#c0392b', 'bold', 'center')
        )
    for x, y, text, size, color, weight, align in lines:
        ax_card.text(
            x, y, text,
            transform=ax_card.transAxes,
            fontsize=size, color=color, fontweight=weight,
            ha=align, va='center'
        )

    fig3.savefig('clinical_fig3_signal_detail.png', dpi=150, bbox_inches='tight',
                 facecolor='#F7F9FC')
    plt.show()

    torch.cuda.empty_cache()
    return fig1, fig2, fig3


# ── Usage ────────────────────────────────────────────────────────────────────
x_temp_b, x_spec_b, y_b, pmask_b = next(iter(val_loader))
fig1, fig2, fig3 = explain_single_sample(
    model, x_temp_b, x_spec_b, pmask_b, y_b, batch_idx=0
)

In [ ]:
!pip install fvcore

In [18]:
from fvcore.nn import FlopCountAnalysis, parameter_count_table

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dummy inputs — batch=1, window=256 epochs
dummy_temp = torch.randn(1, 256, 3000).to(device)
dummy_spec = torch.randn(1, 256, 34).to(device)
dummy_mask = torch.zeros(1, 256, dtype=torch.bool).to(device)

for fusion_type in ('concat', 'gated', 'cross_attn'):
    model = FusedSleepStagingModel(fusion_type=fusion_type).to(device)
    model.eval()

    # ── Params ───────────────────────────────────────────────────────────────
    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ── FLOPs ─────────────────────────────────────────────────────────────────
    # FusedSleepStagingModel.forward returns (logits, aux_temp, aux_spec)
    # fvcore traces the full graph so all three outputs are counted
    flops = FlopCountAnalysis(model, (dummy_temp, dummy_spec, dummy_mask))
    flops.unsupported_ops_warnings(False)
    flops.uncalled_modules_warnings(False)

    print(f"[{fusion_type:<10}]  FLOPs : {flops.total():,}  ({flops.total()/1e9:.3f}G)")
    print(f"[{fusion_type:<10}]  Params: {total_params:,} ({total_params/1e6:.3f}M)  "
          f"Trainable: {trainable_params:,}")
    print()

[concat    ]  FLOPs : 4,843,110,400  (4.843G)
[concat    ]  Params: 1,262,359 (1.262M)  Trainable: 1,262,359



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[gated     ]  FLOPs : 4,842,946,560  (4.843G)
[gated     ]  Params: 1,262,103 (1.262M)  Trainable: 1,262,103



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[cross_attn]  FLOPs : 4,927,488,000  (4.927G)
[cross_attn]  Params: 1,527,063 (1.527M)  Trainable: 1,527,063

